# <font color="#418FDE" size="6.5" uppercase>**Text Features**</font>

>Last update: 20260712.
    
By the end of this Lecture, you will be able to:
- Convert dictionaries and text documents into scikit-learn feature matrices. 
- Configure tokenization, n-grams, vocabulary, stopwords, and frequency controls. 
- Build sparse text pipelines for classification tasks. 


## **1. Dictionary Feature Extraction**

### **1.1. DictVectorizer Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_01.jpg?v=1783909564" width="250">



>* Turns dictionary records into numeric feature matrices
>* Expands named attributes into learnable columns

>* Numeric values stay as single columns
>* Categories become features; missing keys stay absent

>* Maps readable features to matrix columns
>* Uses sparse, consistent matrices for new data



### **1.2. Feature Hashing**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_02.jpg?v=1783909567" width="250">



>* Hash dictionary features into fixed matrix columns
>* No stored vocabulary needed for changing features

>* Handles constantly changing feature names
>* Keeps streaming matrices fixed and memory predictable

>* Hashing maps features into fixed bins
>* Collisions trade interpretability for speed



### **1.3. Hashing Tradeoffs**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_01_03.jpg?v=1783909569" width="250">



>* Hash features into fixed-size matrices
>* Scales well for huge, changing data

>* Hashing collisions can blend unrelated feature signals
>* Feature count balances speed, memory, and accuracy

>* Hashing reduces feature interpretability
>* Use hashing for scalable production workflows



## **2. Text Vectorizers**

### **2.1. Count Vectorizer Basics**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_01.jpg?v=1783909549" width="250">



>* Convert text into numeric word-count features
>* Create document-term matrices for machine learning

>* Bag-of-words counts terms, ignoring word order
>* Simple counts reveal patterns but miss context

>* Sparse matrices store text counts efficiently
>* Consistent vocabularies define reusable feature columns



In [ ]:
#@title Python Code - Count Vectorizer Basics

# Count vectorization turns text into numeric features.
# This example builds a tiny vocabulary manually.
# We inspect counts without using scikit learn.

import re
from collections import Counter

# Tiny documents keep the output readable.
documents = [
    "Free delivery and fast packaging",
    "Delayed delivery damaged the packaging",
    "Free refund for delayed order",
]

# Stopwords remove common low information words.
stopwords = {"and", "the", "for"}

# Tokenization lowercases words and removes punctuation.
def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return [word for word in words if word not in stopwords]

# Build a sorted vocabulary from all documents.
vocabulary = sorted({
    token for doc in documents for token in tokenize(doc)
})

# Count each vocabulary term in each document.
matrix = []
for doc in documents:
    counts = Counter(tokenize(doc))
    row = [counts.get(term, 0) for term in vocabulary]
    matrix.append(row)

# Show the learned feature columns.
print("Vocabulary:", vocabulary)

# Show compact document term count rows.
for index, row in enumerate(matrix, start=1):
    print(f"Document {index} counts:", row)

# Transform new text using the same vocabulary.
new_text = "Delayed refund and free packaging"
new_counts = Counter(tokenize(new_text))
new_row = [new_counts.get(term, 0) for term in vocabulary]

# Unknown words are ignored by this fixed vocabulary.
print("New document counts:", new_row)



### **2.2. Tokenization Controls**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_02.jpg?v=1783909558" width="250">



>* Tokenizers split text into feature units
>* Token choices affect model signal and noise

>* Define valid tokens for each domain
>* Preserve distinctions classifiers need to learn

>* Preprocessing choices affect token meaning
>* Match tokenization to task and data



In [ ]:
#@title Python Code - Tokenization Controls

# Demonstrate tokenization controls with tiny documents.
# Compare default and custom token patterns.
# Keep output short for beginner learners.

# No extra installations are required.

import re
from collections import Counter

# Create tiny documents with tricky text patterns.
documents = [
    "Battery-life is great after 3 days!",
    "Email support@example.com about BP and HR.",
    "Use #Sale2026 for 20% off today.",
]

# Define a simple default-like token pattern.
default_pattern = r"\b\w\w+\b"
custom_pattern = r"[#@]?\b[\w.-]+\b"

# Tokenize documents using a chosen regular expression.
def tokenize_many(texts, pattern, lowercase=True):
    flags = re.IGNORECASE if lowercase else 0
    return [re.findall(pattern, text, flags) for text in texts]

# Build a small vocabulary from tokenized documents.
def build_vocabulary(token_lists, min_count=1):
    counts = Counter(token for row in token_lists for token in row)
    return sorted(token for token, count in counts.items() if count >= min_count)

# Convert token lists into a count matrix.
def count_matrix(token_lists, vocabulary):
    index = {token: position for position, token in enumerate(vocabulary)}
    matrix = [[0 for token in vocabulary] for row in token_lists]
    for row_number, tokens in enumerate(token_lists):
        for token in tokens:
            if token in index:
                matrix[row_number][index[token]] += 1
    return matrix

# Compare tokenization choices on the same documents.
default_tokens = tokenize_many(documents, default_pattern)
custom_tokens = tokenize_many(documents, custom_pattern)

# Build vocabularies after each tokenization choice.
default_vocab = build_vocabulary(default_tokens)
custom_vocab = build_vocabulary(custom_tokens)

# Build one feature matrix from custom tokens.
custom_matrix = count_matrix(custom_tokens, custom_vocab)

# Print compact results for quick inspection.
print("Default first document tokens:", default_tokens[0])
print("Custom first document tokens:", custom_tokens[0])
print("Default vocabulary size:", len(default_vocab))
print("Custom vocabulary size:", len(custom_vocab))
print("Custom vocabulary sample:", custom_vocab[:8])
print("Custom matrix shape:", (len(custom_matrix), len(custom_vocab)))
print("First custom feature row:", custom_matrix[0][:8])



### **2.3. N Gram Features**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_02_03.jpg?v=1783909551" width="250">



>* N grams count short token sequences
>* They capture phrases and local word order

>* Balance phrase detail with feature space size
>* Use frequency controls to reduce noise

>* Phrase wording can reveal domain-specific meaning
>* Control n grams to avoid noisy features



In [ ]:
#@title Python Code - N Gram Features

# Demonstrate n gram text features.
# Use tiny documents for clarity.
# Compare unigrams with bigrams.

from collections import Counter
import re

# Store short example documents.
documents = [
    "good battery life and fast delivery",
    "not good battery life and slow delivery",
    "great customer service and fast delivery",
]

# Define a simple tokenizer.
def tokenize(text):
    words = re.findall(r"[a-z]+", text.lower())
    return words

# Build n grams from tokens.
def make_ngrams(tokens, n_value):
    limit = len(tokens) - n_value + 1
    return [" ".join(tokens[i:i+n_value]) for i in range(limit)]

# Count selected n gram ranges.
def vectorize(texts, ngram_range):
    rows = []
    vocabulary = []

    for text in texts:
        tokens = tokenize(text)
        features = []

        for n_value in range(ngram_range[0], ngram_range[1] + 1):
            features.extend(make_ngrams(tokens, n_value))
        rows.append(Counter(features))

    vocabulary = sorted({term for row in rows for term in row})
    matrix = [[row.get(term, 0) for term in vocabulary] for row in rows]
    return vocabulary, matrix

# Create unigram and bigram matrices.
uni_vocab, uni_matrix = vectorize(documents, (1, 1))
bi_vocab, bi_matrix = vectorize(documents, (1, 2))

# Select phrase features for display.
interesting = ["not good", "battery life", "fast delivery"]
phrase_counts = {term: bi_matrix[1][bi_vocab.index(term)] for term in interesting}

# Print compact teaching results.
print("Documents:", len(documents))
print("Unigram feature count:", len(uni_vocab))
print("Unigram plus bigram feature count:", len(bi_vocab))
print("Example bigram counts in document 2:", phrase_counts)
print("First six bigram vocabulary terms:", bi_vocab[:6])



## **3. TFIDF Text Pipelines**

### **3.1. TFIDF Transformation**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_01.jpg?v=1783909547" width="250">



>* TFIDF converts text into weighted features
>* Highlights distinctive words for classification

>* TFIDF weights distinctive terms after vectorization
>* Balances document lengths to improve classification

>* Sparse TFIDF stores only meaningful nonzero values
>* Efficient, reproducible, but not true language understanding



In [ ]:
#@title Python Code - TFIDF Transformation

# Learn TFIDF using tiny text examples.
# Compare raw counts with weighted features.
# Build a sparse classification style pipeline.

import math
from collections import Counter, defaultdict

# Create tiny labeled support messages.
documents = [
    "refund delivery late refund",
    "password reset login password",
    "delivery package arrived late",
    "subscription cancel refund now",
]

# Store simple labels for context.
labels = ["billing", "account", "shipping", "billing"]

# Tokenize text with lowercase splitting.
tokenized = [text.lower().split() for text in documents]

# Build a sorted vocabulary.
vocabulary = sorted({word for doc in tokenized for word in doc})
word_to_index = {word: i for i, word in enumerate(vocabulary)}

# Count document frequency for each word.
document_frequency = defaultdict(int)
for doc in tokenized:
    for word in set(doc):
        document_frequency[word] += 1

# Compute smoothed inverse document frequency.
number_of_docs = len(tokenized)
idf = {
    word: math.log((1 + number_of_docs) / (1 + document_frequency[word])) + 1
    for word in vocabulary
}

# Build sparse TFIDF rows as dictionaries.
sparse_tfidf = []
for doc in tokenized:
    counts = Counter(doc)
    total_terms = sum(counts.values())
    row = {}
    for word, count in counts.items():
        row[word_to_index[word]] = (count / total_terms) * idf[word]
    sparse_tfidf.append(row)

# Show compact teaching results.
print("Vocabulary:", vocabulary)
print("Matrix shape:", (len(documents), len(vocabulary)))
print("Sparse nonzero counts:", [len(row) for row in sparse_tfidf])
print("Top TFIDF terms per document:")

# Print one strongest term per document.
for i, row in enumerate(sparse_tfidf):
    best_index = max(row, key=row.get)
    best_word = vocabulary[best_index]
    print(f"doc {i}: {labels[i]} -> {best_word} = {row[best_index]:.3f}")



### **3.2. Vocabulary Limits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_02.jpg?v=1783909541" width="250">



>* Limit vocabulary to useful predictive terms
>* Reduce noise, memory use, and training time

>* Keep frequent terms, remove noisy rare ones
>* Filter common terms to improve generalization

>* Limit vocabulary for faster, stable pipelines
>* Balance compact features with important rare terms



### **3.3. Sparse Text Pipeline**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/LFF/Scikit-Learn A to Z/Module_07/Lecture_A/image_03_03.jpg?v=1783909535" width="250">



>* Sparse storage handles huge text feature spaces
>* Pipelines keep classification efficient and reproducible

>* Keep text processing consistent across predictions
>* Prevent mismatched vocabularies and unreliable results

>* Evaluate transformer and classifier together
>* Sparse TFIDF supports scalable classification



# <font color="#418FDE" size="6.5" uppercase>**Text Features**</font>


In this lecture, you learned to:
- Convert dictionaries and text documents into scikit-learn feature matrices. 
- Configure tokenization, n-grams, vocabulary, stopwords, and frequency controls. 
- Build sparse text pipelines for classification tasks. 

In the next Lecture (Lecture B), we will go over 'Images Custom'